# 4장. pandas로 데이터에 질문하기

이 노트북은 데이터를 선택·필터링·정렬하고, 안전하게 병합한 뒤 완료 주문 기준 요약표를 만드는 실습 자료입니다.


## 학습 목표

- 실제 컬럼명과 값의 종류를 확인한 뒤 필터링합니다.
- 수량과 단가로 주문 상세 금액을 계산합니다.
- `validate`와 `indicator`를 사용해 병합 결과를 검증합니다.
- 취소·환불 주문을 제외하고 완료 주문 기준 매출을 계산합니다.
- 개인정보를 최소화한 분석 결과를 저장합니다.


## 0. 제출 정보

- 이름: 권혁수
- GitHub ID: kwonhyeoksoo
- 작성일: 2026-10-07
- 최종 제출 URL: https://github.com/kwonhyeoksoo/llm-data-analysis-study/blob/main/chapter04/chapter04.ipynb

## 1. 프로젝트 루트와 실행 환경 확인

VS Code에서 Notebook을 실행하면 현재 작업 폴더가 프로젝트 루트 또는 `notebooks` 폴더일 수 있습니다. 아래 코드는 상위 폴더를 확인해 프로젝트 루트를 찾습니다.


In [1]:
from pathlib import Path
import sys

import pandas as pd


def find_project_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / 'requirements.txt').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('프로젝트 루트 폴더를 찾을 수 없습니다.')


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print('Python 실행 파일:', sys.executable)
print('현재 작업 폴더:', Path.cwd())
print('프로젝트 루트:', PROJECT_ROOT)
print('데이터 폴더:', DATA_DIR)
print('결과 저장 폴더:', REPORT_DIR)


Python 실행 파일: c:\dev\llm-data-analysis-study\llm-data-analysis-course\.venv\Scripts\python.exe
현재 작업 폴더: c:\dev\llm-data-analysis-study\llm-data-analysis-course\notebooks
프로젝트 루트: C:\dev\llm-data-analysis-study\llm-data-analysis-course
데이터 폴더: C:\dev\llm-data-analysis-study\llm-data-analysis-course\data\raw
결과 저장 폴더: C:\dev\llm-data-analysis-study\llm-data-analysis-course\reports


## 2. 데이터 파일 확인과 불러오기

파일이 없다면 프로젝트 루트에서 `python scripts/generate_sample_data.py`를 먼저 실행하세요.


In [2]:
required_files = ['customers.csv', 'products.csv', 'orders.csv', 'order_items.csv']
missing_files = [name for name in required_files if not (DATA_DIR / name).exists()]

if missing_files:
    raise FileNotFoundError(
        '필요한 데이터 파일이 없습니다: ' + ', '.join(missing_files)
        + '. 프로젝트 루트에서 python scripts/generate_sample_data.py를 실행하세요.'
    )

customers = pd.read_csv(DATA_DIR / 'customers.csv')
products = pd.read_csv(DATA_DIR / 'products.csv')
orders = pd.read_csv(DATA_DIR / 'orders.csv')
order_items = pd.read_csv(DATA_DIR / 'order_items.csv')

print('데이터 불러오기 완료')


데이터 불러오기 완료


## 3. 데이터 구조와 필수 컬럼 확인

LLM이 작성한 코드가 실제 컬럼명과 일치하는지 먼저 확인합니다.


In [3]:
datasets = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
}

expected_columns = {
    'customers': ['customer_id', 'gender', 'age', 'city'],
    'products': ['product_id', 'product_name', 'category', 'price'],
    'orders': ['order_id', 'customer_id', 'order_date', 'order_status'],
    'order_items': ['order_id', 'product_id', 'quantity', 'unit_price'],
}

for name, df in datasets.items():
    missing = [col for col in expected_columns[name] if col not in df.columns]
    print(name, df.shape, df.columns.tolist())
    if missing:
        raise KeyError(f'{name}에 필요한 컬럼이 없습니다: {missing}')


customers (150, 6) ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
products (100, 4) ['product_id', 'product_name', 'category', 'price']
orders (300, 5) ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']
order_items (764, 5) ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']


## 1. 질문과 필요한 데이터 선택

### 내가 확인하려는 질문

완료(completed)된 주문을 기준으로, 어떤 카테고리/상품/월/고객이 매출에 가장 크게 기여하는지 확인하고 싶다.

### 사용한 파일/컬럼

- customers.csv: customer_id, gender, age, city
- products.csv: product_id, product_name, category, price
- orders.csv: order_id, customer_id, order_date, order_status
- order_items.csv: order_id, product_id, quantity, unit_price

### 결과 관찰

4개 CSV 파일이 모두 오류 없이 로드되었다. customers는 150행 6개 컬럼, products는 100행 4개 컬럼, orders는 300행 5개 컬럼, order_items는 764행 5개 컬럼으로 구성되어 있었다. 분석에 필요한 필수 컬럼(customer_id, product_id, quantity, unit_price, order_status 등)이 모두 존재함을 확인했다.

### 나의 해석과 판단

매출 분석을 하려면 order_items(수량/단가)와 orders(주문 상태/날짜)를 연결해야 하고, 여기에 products(카테고리), customers(고객 속성)를 추가로 병합해야 한다고 판단했다. 코드를 작성하기 전에 df.columns.tolist()로 실제 컬럼명을 먼저 확인한 것은, 추측이나 LLM이 제안한 컬럼명이 실제와 다를 경우 바로 KeyError로 이어질 수 있기 때문이다.

### 업무·분석적 의미

실제 데이터 분석에서는 문서화가 부실하거나 컬럼명이 예상과 다른 경우가 흔하기 때문에, 분석 코드를 작성하기 전에 실제 데이터 구조를 먼저 확인하는 습관이 불필요한 시행착오를 줄여준다.

### 한계와 추가 확인 사항

아직 각 컬럼의 실제 값(예: order_status에 어떤 값들이 있는지, city 표기가 어떻게 되어 있는지)은 확인하지 않았다. 이는 다음 단계인 필터링에서 확인할 예정이다.

## 4. 컬럼 선택과 행 필터링

이 저장소의 샘플 데이터는 도시명을 `서울`, `부산`처럼 한글로 생성합니다. 필터링 전에 실제 값을 확인합니다.


In [4]:
customer_basic = customers[['customer_id', 'gender', 'age', 'city']]
display(customer_basic.head())

print('도시별 고객 수')
display(customers['city'].value_counts())

customers_over_30 = customers[customers['age'] >= 30]
seoul_customers = customers[customers['city'] == '서울']
city_customers = customers[customers['city'].isin(['서울', '부산'])]

print('30세 이상 고객 수:', len(customers_over_30))
print('서울 고객 수:', len(seoul_customers))
print('서울 또는 부산 고객 수:', len(city_customers))


,customer_id,gender,age,city
0,1,F,19,광주
1,2,F,32,대구
2,3,F,61,성남
3,4,F,55,울산
4,5,F,19,부산


도시별 고객 수


city
성남    21
광주    17
부산    16
대구    15
서울    15
울산    14
인천    14
대전    14
수원    13
고양    11
Name: count, dtype: int64

30세 이상 고객 수: 111
서울 고객 수: 15
서울 또는 부산 고객 수: 31


## 5. 정렬과 주문 상태 확인


In [5]:
display(products.sort_values('price', ascending=False).head(10))
display(customers.sort_values('age', ascending=False).head(10))

print('주문 상태별 건수')
display(orders['order_status'].value_counts())


,product_id,product_name,category,price
98,99,뷰티 상품 099,뷰티,200000
69,70,패션 상품 070,패션,198000
57,58,식품 상품 058,식품,197000
42,43,뷰티 상품 043,뷰티,197000
23,24,스포츠 상품 024,스포츠,196000
8,9,스포츠 상품 009,스포츠,193000
36,37,뷰티 상품 037,뷰티,193000
71,72,뷰티 상품 072,뷰티,189000
7,8,스포츠 상품 008,스포츠,189000
52,53,생활용품 상품 053,생활용품,188000


,customer_id,name,gender,age,city,signup_date
14,15,장정식,M,69,서울,2026-08-12
8,9,송지민,M,69,서울,2025-12-27
54,55,윤영철,M,69,광주,2023-10-08
78,79,이서준,M,69,부산,2023-09-21
136,137,김선영,M,68,울산,2025-02-05
45,46,김서현,M,67,대전,2026-07-21
132,133,김성진,M,67,성남,2026-08-06
123,124,김시우,M,67,대구,2024-12-02
11,12,김정남,F,66,대전,2024-12-08
57,58,김성훈,F,66,성남,2025-08-22


주문 상태별 건수


order_status
completed    184
cancelled     64
refunded      52
Name: count, dtype: int64

## 6. 주문 상세 금액 만들기

`line_total`은 주문 상세 1행의 금액입니다. 주문 상태를 연결하기 전 합계는 확정 매출이 아니라 전체 주문 상세 금액입니다.


In [6]:
order_items = order_items.copy()
order_items['line_total'] = order_items['quantity'] * order_items['unit_price']

all_order_amount = order_items['line_total'].sum()
print('전체 주문 상세 금액:', all_order_amount)
display(order_items[['order_id', 'product_id', 'quantity', 'unit_price', 'line_total']].head())


전체 주문 상세 금액: 255610000


,order_id,product_id,quantity,unit_price,line_total
0,1,100,3,102000,306000
1,1,87,5,25000,125000
2,1,7,3,142000,426000
3,1,9,3,193000,579000
4,2,72,4,189000,756000


## 2. 필터링·정렬·파생 컬럼

- 적용한 필터 조건: age >= 30 (30세 이상), city.isin(['서울', '부산'])
- 정렬 기준: products는 price 내림차순, customers는 age 내림차순
- 만든 파생 컬럼: line_total (주문 상세 1건의 금액)
- `line_total` 계산식: quantity * unit_price

![필터와 파생 컬럼](images/step02_transform.png)

### 결과 관찰

도시별 고객 수를 확인한 결과 30세 이상 고객은 111명, 서울 고객은 15명, 서울 또는 부산 고객은 31명이었다. 가격 기준 상위 상품은 대부분 뷰티 및 패션 카테고리였고, 나이 기준 상위 고객은 60대 후반에 집중되어 있었다. 주문 상태는 completed 184건, cancelled 64건, refunded 52건으로 나뉘어 있어 전체 주문 중 일부는 완료되지 않았음을 확인했다. order_items에 quantity * unit_price로 line_total을 계산한 결과 전체 주문 상세 금액 합계는 255,610,000원이었다. 다만 이 금액은 주문 상태를 아직 반영하지 않은 전체 합계이다.

### 나의 해석과 판단

필터링 전에 value_counts()로 실제 값을 먼저 확인한 것은, 예를 들어 '서울'이 아니라 다른 표기로 저장되어 있었다면 필터 결과가 0건이 되어도 원인을 모른 채 지나칠 수 있기 때문이다. 또한 order_status에 completed 외에 cancelled, refunded가 섞여 있다는 것을 확인했기 때문에, line_total의 단순 합계인 255,610,000원을 그대로 매출이라고 부를 수 없다고 판단했다. 이후 단계에서 반드시 completed 주문만 걸러서 다시 계산해야 한다.

### 업무·분석적 의미

필터 기준(연령, 도시, 주문 상태)에 따라 매출이나 고객 수 같은 핵심 지표가 크게 달라질 수 있다. 예를 들어 취소/환불 건까지 포함한 금액을 매출로 보고하면 실제보다 과대 집계된 수치를 전달하게 되므로 분석 목적에 맞는 필터 기준을 명확히 하고 그 기준을 분석 결과에 함께 명시하는 것이 중요하다.

### 한계와 추가 확인 사항

현재 계산한 전체 주문 상세 금액은 주문 상태를 반영하지 않은 값이므로, 다음 단계에서 completed 주문만 필터링한 뒤 다시 계산한 금액과 반드시 비교해야 한다.

## 7. 주문 데이터 병합과 검증

`validate='many_to_one'`은 주문 상세의 동일 주문 ID는 여러 번 나올 수 있지만 주문 테이블의 주문 ID는 한 번만 나와야 한다는 뜻입니다.


In [7]:
print('orders.order_id 중복 수:', orders['order_id'].duplicated().sum())

order_sales = order_items.merge(
    orders[['order_id', 'customer_id', 'order_date', 'order_status']],
    on='order_id',
    how='left',
    validate='many_to_one',
    indicator=True,
)

print('병합 전 행 수:', len(order_items))
print('병합 후 행 수:', len(order_sales))
display(order_sales['_merge'].value_counts())

order_sales = order_sales.drop(columns='_merge')


orders.order_id 중복 수: 0
병합 전 행 수: 764
병합 후 행 수: 764


_merge
both          764
left_only       0
right_only      0
Name: count, dtype: int64

## 8. 완료 주문만 선택하기

이후의 매출 요약은 `order_status == 'completed'`인 주문만 사용합니다.


In [8]:
order_sales['order_date'] = pd.to_datetime(order_sales['order_date'], errors='coerce')
print('날짜 변환 실패:', order_sales['order_date'].isna().sum())

completed_order_sales = order_sales[
    order_sales['order_status'] == 'completed'
].copy()

completed_order_sales['order_month'] = (
    completed_order_sales['order_date'].dt.to_period('M').astype(str)
)

print('전체 주문 상세 행 수:', len(order_sales))
print('완료 주문 상세 행 수:', len(completed_order_sales))
print('완료 주문 매출:', completed_order_sales['line_total'].sum())


날짜 변환 실패: 0
전체 주문 상세 행 수: 764
완료 주문 상세 행 수: 474
완료 주문 매출: 148990000


## 9. 상품 데이터 병합과 검증


In [9]:
print('products.product_id 중복 수:', products['product_id'].duplicated().sum())

completed_sales_items = completed_order_sales.merge(
    products,
    on='product_id',
    how='left',
    validate='many_to_one',
    indicator=True,
)

print('병합 전 행 수:', len(completed_order_sales))
print('병합 후 행 수:', len(completed_sales_items))
display(completed_sales_items['_merge'].value_counts())

completed_sales_items = completed_sales_items.drop(columns='_merge')


products.product_id 중복 수: 0
병합 전 행 수: 474
병합 후 행 수: 474


_merge
both          474
left_only       0
right_only      0
Name: count, dtype: int64

## 3. merge 검증

### (1) order_items + orders

- 병합한 데이터: order_items + orders
- 사용한 key: order_id
- `validate` 결과: many_to_one 조건 충족 (오류 없음)
- `indicator` 결과: _merge가 전부 both (left_only 0건, right_only 0건)
- 병합 전/후 행 수: 764 / 764

### (2) completed_order_sales + products

- 병합한 데이터: completed_order_sales + products
- 사용한 key: product_id
- `validate` 결과: many_to_one 조건 충족 (오류 없음)
- `indicator` 결과: _merge가 전부 both (left_only 0건, right_only 0건)
- 병합 전/후 행 수: 474 / 474

![merge 검증](images/step03_merge.png)

### 결과 관찰

두 번의 병합 모두 병합 전후 행 수가 그대로 유지되었고 _merge 값이 전부 both로 나왔다. 첫 번째 병합(order_items + orders)에서는 orders.order_id 중복이 0건, 두 번째 병합(completed_order_sales + products)에서는 products.product_id 중복이 0건임을 먼저 확인했고 이후 병합 결과에서도 매칭 실패(left_only, right_only)가 전혀 없었다.

### 나의 해석과 판단

이번 두 병합이 모두 안전하다고 판단한 근거는, 병합 전에 각 기준 테이블(orders, products)의 key가 중복되지 않는다는 것을 먼저 확인했고 병합 후에도 행 수 변화와 매칭 실패가 전혀 없었기 때문이다. validate='many_to_one' 옵션 덕분에, 만약 key가 중복되어 있었다면 병합 시점에 바로 오류가 발생해 문제를 사전에 발견할 수 있었을 것이다.

### 업무·분석적 의미

key가 중복된 상태로 병합하면 행 수가 의도치 않게 늘어나면서 집계 금액이 실제보다 부풀려질 수 있다. 이번 실습처럼 두 번 연속으로 병합하는 경우, 각 단계마다 검증을 생략하면 어느 단계에서 문제가 생겼는지 찾기 어려워진다. 매 병합마다 validate와 indicator로 검증하는 것이 분석 결과의 신뢰성을 지키는 최소한의 절차라고 생각한다.

### 한계와 추가 확인 사항

이번 실습 데이터에서는 두 병합 모두 매칭 실패가 없었지만 실제 데이터에서는 외래키 결측이나 오타로 인한 left_only가 발생할 수 있다. 이 경우 어떤 행이 매칭되지 않았는지 찾아 원인을 분석하는 절차는 이번 실습에서는 다루지 않았다.

## 10. 카테고리별·상품별 매출


In [10]:
category_sales = (
    completed_sales_items
    .groupby('category', as_index=False)
    .agg(
        total_quantity=('quantity', 'sum'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

category_sales['sales_ratio'] = (
    category_sales['total_sales'] / category_sales['total_sales'].sum() * 100
).round(2)

display(category_sales)

product_sales = (
    completed_sales_items
    .groupby(['product_id', 'product_name', 'category'], as_index=False)
    .agg(
        total_quantity=('quantity', 'sum'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

display(product_sales.head(10))


,category,total_quantity,total_sales,sales_ratio
3,스포츠,295,31743000,21.31
5,전자기기,259,26400000,17.72
2,생활용품,272,23915000,16.05
1,뷰티,223,23383000,15.69
4,식품,133,16573000,11.12
0,도서,149,16389000,11.00
6,패션,111,10587000,7.11


,product_id,product_name,category,total_quantity,total_sales
39,41,스포츠 상품 041,스포츠,35,5705000
11,12,식품 상품 012,식품,25,4375000
8,9,스포츠 상품 009,스포츠,20,3860000
70,72,뷰티 상품 072,뷰티,20,3780000
69,71,전자기기 상품 071,전자기기,23,3703000
66,68,스포츠 상품 068,스포츠,26,3640000
78,81,전자기기 상품 081,전자기기,22,3630000
10,11,패션 상품 011,패션,31,3565000
20,22,생활용품 상품 022,생활용품,29,3248000
86,89,생활용품 상품 089,생활용품,30,3090000


## 11. 월별 매출


In [11]:
monthly_summary = (
    completed_order_sales
    .groupby('order_month', as_index=False)
    .agg(
        total_sales=('line_total', 'sum'),
        order_count=('order_id', 'nunique'),
    )
    .sort_values('order_month')
)

monthly_summary['average_order_value'] = (
    monthly_summary['total_sales'] / monthly_summary['order_count']
).round(0)

display(monthly_summary)


,order_month,total_sales,order_count,average_order_value
0,2025-09,2650000,5,530000.0
1,2025-10,16970000,19,893158.0
2,2025-11,11021000,13,847769.0
3,2025-12,21085000,23,916739.0
4,2026-01,14272000,16,892000.0
5,2026-02,4784000,7,683429.0
6,2026-03,18323000,25,732920.0
7,2026-04,10254000,15,683600.0
8,2026-05,16831000,19,885842.0
9,2026-06,12994000,17,764353.0


## 12. 고객별 구매 금액

고객 ID로 먼저 집계한 뒤 고객 속성을 붙입니다. 출력에는 실제 이름 대신 익명화된 고객 라벨을 사용합니다.


In [12]:
customer_sales = (
    completed_order_sales
    .groupby('customer_id', as_index=False)
    .agg(
        order_count=('order_id', 'nunique'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

customer_sales = customer_sales.merge(
    customers[['customer_id', 'city']],
    on='customer_id',
    how='left',
    validate='one_to_one',
)

customer_sales['customer_label'] = 'Customer ' + customer_sales['customer_id'].astype(str)
display(customer_sales[['customer_label', 'city', 'order_count', 'total_sales']].head(10))


,customer_label,city,order_count,total_sales
0,Customer 117,성남,5,4100000
1,Customer 102,고양,4,3996000
2,Customer 83,수원,4,3880000
3,Customer 30,서울,5,3590000
4,Customer 40,서울,4,3523000
5,Customer 20,인천,2,3191000
6,Customer 3,성남,2,3178000
7,Customer 111,광주,3,3153000
8,Customer 66,서울,4,3093000
9,Customer 147,부산,2,2990000


## 4. completed 주문 범위와 집계

- 분석 범위 정의: order_status == 'completed'인 주문 상세만 사용 (전체 764행 중 474행)
- 카테고리별 결과: 스포츠(31,743,000원, 21.31%)가 1위, 이어서 전자기기(17.72%), 생활용품(16.05%)
- 상품별 결과: '스포츠 상품 041'이 5,705,000원으로 단일 상품 매출 1위
- 월별 결과: 2025-12가 21,085,000원으로 최고 매출 월, 2025-09가 2,650,000원으로 최저
- 고객별 결과: Customer 117이 4,100,000원으로 구매액 1위

![핵심 집계 결과](images/step04_groupby.png)

### 결과 관찰

날짜 변환 실패는 0건이었고, completed 주문만 필터링하자 전체 764건의 주문 상세 중 474건(62%)만 남았다. completed 주문 기준 매출 합계는 148,990,000원으로, 필터링 전 전체 합계의 약 58% 수준이었다. 카테고리별로는 스포츠(21.31%), 전자기기(17.72%), 생활용품(16.05%) 순으로 매출 비중이 높았고, 패션(7.11%)이 가장 낮았다. 월별로는 2025-12가 21,085,000원으로 가장 높았고 2025-09가 2,650,000원으로 가장 낮아 월별 편차가 큰 편이었다. 고객별로는 상위 고객(Customer 117)의 총 구매액이 4,100,000원으로, 상위 몇 명의 고객이 상대적으로 높은 금액을 차지하고 있었다.

### 나의 해석과 판단

필터링 전 전체 주문 상세 금액과 completed 주문만 걸러낸 금액 사이에 상당한 차이가 있다는 것이 가장 중요한 발견이라고 판단했다. 이는 주문 상태를 고려하지 않고 단순히 line_total을 합산하면 취소 및 환불 건까지 포함되어 실제보다 매출이 과대 집계된다는 것을 보여준다. 카테고리별 결과에서는 스포츠 카테고리가 매출과 판매량 모두 1위를 차지했는데, 이는 단가가 특별히 높아서라기보다 판매량 자체가 많았기 때문으로 해석했다.

### 업무·분석적 의미

월별 매출 편차가 크다는 것은 계절성이나 특정 이벤트(프로모션 등)의 영향을 받았을 가능성을 시사하므로, 다음 분석에서는 월별 추이의 원인을 더 들여다볼 필요가 있다고 판단했다. 또한 상위 고객 몇 명의 구매액이 두드러지는 패턴은 향후 VIP 고객 관리나 타겟 마케팅 전략을 세울 때 참고할 수 있는 기초 자료가 될 수 있다.

### 한계와 추가 확인 사항

`total_sales`는 completed 주문에 포함된 line_total의 합계일 뿐, 할인, 배송비, 환불 수수료 등을 반영한 회계상 순매출과는 다르다. 따라서 이 수치를 그대로 회사의 공식 매출로 보고할 수는 없으며, 실제 재무 지표와 대조하는 과정이 추가로 필요하다.

## 13. 결과 저장하기


In [13]:
outputs = {
    'ch04_category_sales.csv': category_sales,
    'ch04_product_sales.csv': product_sales,
    'ch04_monthly_sales.csv': monthly_summary,
    'ch04_customer_sales.csv': customer_sales,
}

for filename, df in outputs.items():
    path = REPORT_DIR / filename
    df.to_csv(path, index=False, encoding='utf-8-sig')
    print(filename, path.exists(), path.stat().st_size)


ch04_category_sales.csv True 254
ch04_product_sales.csv True 4409
ch04_monthly_sales.csv True 443
ch04_customer_sales.csv True 3387


In [14]:
print('completed line_total 합계:', completed_order_sales['line_total'].sum())
print('category_sales 합계:', category_sales['total_sales'].sum())
print('monthly_summary 합계:', monthly_summary['total_sales'].sum())
print('customer_sales 합계:', customer_sales['total_sales'].sum())

completed line_total 합계: 148990000
category_sales 합계: 148990000
monthly_summary 합계: 148990000
customer_sales 합계: 148990000


## 5. 총합 일치 검증

- 원본 completed `line_total` 합계: 148,990,000원
- 카테고리 합계: 148,990,000원
- 월별 합계: 148,990,000원
- 고객별 합계: 148,990,000원
- 차이 여부: 없음 (4개 수치 모두 정확히 일치)

![총합 검증](images/step05_total_check.png)

### 나의 해석과 판단

completed_order_sales의 line_total 원본 합계와, 이를 카테고리/월/고객 기준으로 각각 groupby한 뒤 다시 합산한 값이 모두 148,990,000원으로 정확히 일치했다. 이는 집계 과정에서 행이 누락되거나 중복 집계된 것이 없다는 것을 의미한다. 만약 병합 과정에서 중복 행이 생겼거나 필터링 조건이 잘못 적용되었다면, 집계 기준에 따라 합계가 서로 다르게 나왔을 것이다.

총합 검증이 필요한 이유는, groupby와 merge를 여러 번 거치는 과정에서 의도치 않게 행이 늘어나거나 줄어드는 실수가 발생하기 쉽기 때문이다. 각 집계표의 출력만 봐서는 이런 문제를 알아차리기 어렵지만 원본 합계와 비교하면 바로 드러난다.

만약 불일치가 발생했다면 다음 순서로 원인을 확인했을 것이다.
1. 각 merge 단계에서 _merge 값에 left_only/right_only가 있었는지 재확인
2. groupby 기준 컬럼에 결측치가 있어 일부 행이 그룹에서 누락되지 않았는지 확인
3. 필터링 조건(order_status == 'completed')이 집계 전 단계에서 일관되게 적용되었는지 확인

## 14. LLM 코드 검증 연습

```text
LLM이 다음 코드를 제안했습니다.

category_sales = order_items.groupby('category')['line_total'].sum()

현재 데이터에서 이 코드가 바로 실행 가능한지 검토해 주세요.
category 컬럼의 위치, 주문 상태 필터링, merge validate와 indicator를 포함해
안전한 수정 코드와 검증 순서를 설명해 주세요.
```


## 6. LLM pandas 코드 검증

- LLM Prompt 요약: order_items에서 카테고리별 line_total 합계를 구하는 코드가 실행 가능한지 검토하고 필요하면 안전한 수정 코드와 검증 순서를 요청함
- 제안 코드 요약: category_sales = order_items.groupby('category')['line_total'].sum()
- 실제 컬럼/범위와 맞지 않은 부분:
1. order_items에는 category 컬럼이 없음 (category는 products 테이블에만 존재) → KeyError 발생
2. order_items는 주문 상태(completed/cancelled/refunded)를 전혀 걸러내지 않은 원본 상태
3. products, orders와의 merge 과정과 validate/indicator 검증이 전혀 없음
- 수정한 내용: orders merge(validate='many_to_one', indicator=True) → completed 필터링 → products merge(validate='many_to_one', indicator=True) → groupby('category')로 수정. 실제로 우리가 섹션 7~9에서 수행한 순서와 동일.
- 최종 판단: 수정 후 사용

![LLM 코드 검증](images/step06_llm_validation.png)

### 나의 해석과 판단

LLM이 제안한 코드는 category라는 그럴듯한 컬럼명을 사용했지만, 실제로는 order_items가 아니라 products 테이블에 속한 컬럼이라는 점을 고려하지 않았다. 또한 주문 상태 필터링과 병합 검증 과정을 모두 생략했기 때문에 설령 컬럼 에러를 수정해서 실행된다 하더라도 취소 및 환불 주문까지 포함된 잘못된 매출 집계를 낼 수 있었다. 코드가 "실행된다"는 것과 "분석 목적에 맞게 올바른 값을 낸다"는 것은 전혀 다른 문제라는 것을 이번 검증을 통해 확인했다. 실제로 우리가 직접 실행한 결과에서도 completed 필터링 전후 매출 합계가 255,610,000원과 148,990,000원으로 크게 달랐는데, LLM이 제안한 코드는 이 차이를 전혀 반영하지 못하는 코드였다.

## 15. 실습 과제

1. 40세 이상 고객을 추출합니다.
2. 상품 가격이 낮은 순서대로 10개를 출력합니다.
3. 결제수단별 주문 수와 비율을 계산합니다.
4. 카테고리별 평균 상품 가격을 계산합니다.
5. 완료 주문과 전체 주문의 금액 차이를 계산합니다.
6. 병합 검증을 포함한 고객별 구매 금액 코드 요청 프롬프트를 작성합니다.


In [15]:
# 과제 코드를 아래에 작성하세요.
# 1. 40세 이상 고객
customers_40_plus = customers[customers['age'] >= 40]
print('40세 이상 고객 수:', len(customers_40_plus))

# 2. 상품 가격 낮은 순 10개
print(products.sort_values('price').head(10))

# 3. 결제수단별 주문 수와 비율
payment_counts = orders['payment_method'].value_counts()
payment_ratio = (payment_counts / len(orders) * 100).round(2)
print(payment_counts)
print(payment_ratio)

# 4. 카테고리별 평균 상품 가격
print(products.groupby('category')['price'].mean().round(0))

# 5. 완료 주문 vs 전체 주문 금액 차이
all_amount = order_items['line_total'].sum()
completed_amount = completed_order_sales['line_total'].sum()
print('전체 주문 금액:', all_amount)
print('완료 주문 금액:', completed_amount)
print('차이:', all_amount - completed_amount)

# 6. 병합 검증 포함 고객별 구매 금액 코드 요청 프롬프트 (텍스트로만 작성)
prompt_example = """
orders와 order_items, customers를 사용해 고객별 구매 금액을 계산하려고 합니다.
다음을 포함해 주세요: line_total 계산, orders merge(validate/indicator),
completed 필터링, customer_id 기준 집계, customers와 merge(validate='one_to_one').
"""
print(prompt_example)

40세 이상 고객 수: 78
    product_id product_name category  price
5            6  전자기기 상품 006     전자기기   5000
27          28    뷰티 상품 028       뷰티   5000
97          98   스포츠 상품 098      스포츠  10000
46          47  생활용품 상품 047     생활용품  11000
94          95  전자기기 상품 095     전자기기  20000
49          50    뷰티 상품 050       뷰티  23000
82          83  전자기기 상품 083     전자기기  24000
86          87    도서 상품 087       도서  25000
41          42    패션 상품 042       패션  28000
58          59    뷰티 상품 059       뷰티  32000
payment_method
kakao_pay        79
naver_pay        77
bank_transfer    74
card             70
Name: count, dtype: int64
payment_method
kakao_pay        26.33
naver_pay        25.67
bank_transfer    24.67
card             23.33
Name: count, dtype: float64
category
도서      106857.0
뷰티      117688.0
생활용품     96438.0
스포츠     111579.0
식품      137143.0
전자기기    101588.0
패션      115909.0
Name: price, dtype: float64
전체 주문 금액: 255610000
완료 주문 금액: 148990000
차이: 106620000

orders와 order_items, customers를 사

## 정리

이번 장에서는 실제 값 확인, 컬럼 선택, 필터링, 정렬, 파생 컬럼, 병합 검증, 완료 주문 기준 집계, CSV 저장 과정을 수행했습니다. 다음 장에서는 결측치, 중복, 타입 오류, 날짜 형식, 이상값 후보를 다루는 데이터 전처리로 이어집니다.


## 7. Chapter 04 최종 인사이트

### 가장 의미 있다고 생각한 결과 2가지

1. completed 주문만 필터링했을 때와 전체 주문을 그대로 합산했을 때 매출 차이가 매우 크다는 점 
2. 스포츠 카테고리가 매출과 판매량 모두 1위를 차지했지만 반대로 월별 매출은 2025-09와 2025-12 사이에 약 8배 차이가 날 정도로 편차가 컸다는 점

### 그 결과를 뒷받침하는 수치/표

- 전체 주문 상세 금액: 255,610,000원 (order_status 미반영)
- completed 주문 매출: 148,990,000원 (전체 764건 중 474건만 해당)
- 카테고리별 매출 1위: 스포츠 31,743,000원(21.31%), 2위 전자기기 17.72%
- 월별 매출 최고: 2025-12 21,085,000원 / 최저: 2025-09 2,650,000원
- 총합 일치 검증: completed 원본 합계, 카테고리/월/고객 합계 모두 148,990,000원으로 정확히 일치

### 추가로 확인하고 싶은 질문

- 취소(cancelled)와 환불(refunded) 주문이 특정 카테고리나 결제수단에 더 집중되어 있는지
- 월별 매출 편차(2025-09 vs 2025-12)가 특정 프로모션이나 시즌 효과 때문인지
- 상위 고객(Customer 117 등)이 특정 카테고리를 반복 구매하는 경향이 있는지

### 현재 결과의 한계

이번 분석은 line_total(quantity × unit_price)을 completed 기준으로 합산한 값이며 할인/배송비/환불 수수료 등을 반영하지 않은 수치이므로 회계상 순매출과 동일하다고 볼 수 없다. 또한 데이터 자체의 품질 (결측치, 이상치 등)은 아직 검증하지 않았다. 

## 최종 제출 체크
- [x] 핵심 셀 Output이 남아 있습니다.
- [x] merge와 총합 검증 Evidence가 있습니다.
- [x] 결과 관찰과 해석이 구분되어 있습니다.
- [x] LLM 코드를 검증했습니다.
- [x] 개인정보/Secret이 없습니다.
- [x] `chapter04/chapter04.ipynb`가 GitHub에서 정상 표시됩니다.
- [x] 최종 Notebook 파일 URL을 제출합니다.